# Learned Cross-Field Quad Remeshing

**Yosuke Kobayashi（小林洋介）**
https://yosuke4061.com/

1 October 2026

Run the cells from top to bottom. Use a GPU runtime with Python 3.13. Do not change the runtime after you start. This notebook does not use ComfyUI.

## What it does

A triangle mesh (GLB) becomes a quad mesh (OBJ). The new edges follow a cross field on the surface: two directions at 90 degrees. Turning that cross by 90 degrees does not change it. This is a 4-RoSy field [3].

The notebook expects a mesh that is already in reasonable shape. A raw generated mesh is not the target. Before training it counts skinny triangles, faces whose height is under 2% of their longest edge. If more than 0.15% of the faces are that thin, the run stops. Cleaned meshes used with this project sit under that line.

## How the field is learned

A small graph network [4] learns the cross on this one mesh. There is no hand-labeled training set. The losses come from the mesh itself: principal curvature, smooth neighbors, creases, boundaries, symmetry, and color edges when color exists. Training is phased. It stops before 200 epochs when the loss stops improving.

The network sits on the mesh graph. Curvature comes from a local surface fit, and the quads come from Instant Meshes [2].

## How the quads are built

A clean mesh with more than 790,000 faces is reduced to 790,000 for the field and for Instant Meshes. The quad count is computed from that reduced face count, not from the original face count. Quad vertices are then moved back onto the original surface, so a dense input still contributes its shape.

Instant Meshes needs edges short enough for its grid. It splits only edges longer than half the target edge. That is about one split on a uniform mesh. If Instant Meshes returns no faces, the run stops. A few missed faces can still be covered by splitting each leftover triangle into three quads. Those quads fill the gap. They do not follow the cross field. Long boundary loops are left open.

Vertex color is an extra guide, not a requirement. Stored vertex colors (`COLOR_0`) are used even when the UV coordinates are a single constant value. If the file has a texture but no vertex colors, a later cell copies the texture onto the vertices first. If it has neither, the quad mesh is still written and vertex colors are skipped. The uploaded file is kept for the vertex colors.

## Steps

1. Stop if the mesh is too skinny. Otherwise repair it so neighboring faces can be walked.
2. Estimate principal curvature directions.
3. Train the graph net, up to 200 epochs. It can stop early.
4. Instant Meshes builds the quads and pulls them back onto the original surface.
5. A few missed faces are tried again, then split into three quads each. An empty extract stops the run.
6. Copy color from the original GLB onto the quad vertices. The result is a quad OBJ with vertex colors. No materials and no textures. Bake PBR yourself in Blender from that OBJ and the original GLB.

## References

[1] Q. Dong, H. Wen, R. Xu, S. Chen, J. Zhou, S. Xin, C. Tu, T. Komura, and W. Wang. NeurCross: A neural approach to computing cross fields for quad mesh generation. *ACM Transactions on Graphics* (SIGGRAPH), 2025. arXiv:2405.13745.

[2] W. Jakob, M. Tarini, D. Panozzo, and O. Sorkine-Hornung. Instant field-aligned meshes. *ACM Transactions on Graphics*, 34(6), 2015.

[3] N. Ray, B. Vallet, W. C. Li, and B. Lévy. N-symmetry direction field design. *ACM Transactions on Graphics*, 27(2), 2008.

[4] W. L. Hamilton, R. Ying, and J. Leskovec. Inductive representation learning on large graphs. In *NeurIPS*, 2017.


In [ ]:
import sys
print(sys.version)
assert sys.version_info[:2] == (3, 13), "This runtime must be Python 3.13. The shipped extractor is cpython-313."
import torch
if torch.cuda.is_available():
    print("device", torch.cuda.get_device_name(0))
else:
    print("device cpu")


## Libraries

The next cell installs trimesh, SciPy, Pillow, PyMeshLab, and Numba. PyTorch is already on Colab.

PyTorch runs the graph net. Numba speeds up the curvature fit. trimesh reads the GLB. SciPy finds nearby triangles. PyMeshLab repairs the mesh.


In [ ]:
import subprocess, sys
subprocess.check_call([sys.executable, "-m", "pip", "install", "-q",
    "trimesh", "scipy", "Pillow", "pymeshlab", "numba"])
print("libraries ready")


## Code download

The next cell downloads the Python code, the viewer, and the Instant Meshes library `im_gnn_bridge.cpython-313-x86_64-linux-gnu.so`. That file is for Linux and Python 3.13 only.

The next cell downloads `brief153_colab.zip` from the GitHub repo.


In [ ]:
import shutil, urllib.request, zipfile
from pathlib import Path
url = "https://raw.githubusercontent.com/koba4061/triangle-to-quad-remeshing/main/brief153_colab.zip"
zpath = Path("/content/brief153_colab.zip")
urllib.request.urlretrieve(url, zpath)
root = Path("/content/brief153")
if root.exists():
    shutil.rmtree(root)
root.mkdir()
with zipfile.ZipFile(zpath) as z:
    z.extractall(root)
so = list((root / "gnn_quad_retopo").glob("im_gnn_bridge*.so"))
print([p.name for p in so])
assert so, "The archive has no im_gnn_bridge shared library."


## Input

Choose a sample from the list, or choose `upload` and send your own `.glb`. Later cells read `/content/in.glb` and do not change that file. A sample is taken from the notebook archive when it is there, otherwise from the GitHub repo.


In [ ]:
import shutil
import urllib.request
from pathlib import Path
MODEL = "upload"  #@param ["upload", "01_starlight_hourglass.glb", "02_obsidian_mask_glow.glb", "03_golden_scale_dragon.glb", "04_melting_clock_porcelain.glb", "05_spiral_ruin_tower.glb", "06_crystal_wing_angel.glb", "07_butterfly_wing_jewelry_box.glb", "08_bronze_ribbon_sculpture.glb", "out_2f6c86d9.glb", "out_8eeec1fa.glb"]
src = Path("/content/in.glb")
if MODEL == "upload":
    from google.colab import files
    got = files.upload()
    names = [n for n in got if n.lower().endswith(".glb")]
    assert len(names) == 1, "Upload exactly one .glb file."
    src.write_bytes(got[names[0]])
    stem = Path(names[0]).stem
    print("uploaded", names[0], src.stat().st_size)
else:
    local = Path("/content/brief153/inputs") / MODEL
    if local.is_file():
        shutil.copyfile(local, src)
        print("sample", MODEL, src.stat().st_size)
    else:
        url = "https://raw.githubusercontent.com/koba4061/triangle-to-quad-remeshing/main/inputs/" + MODEL
        urllib.request.urlretrieve(url, src)
        print("downloaded", MODEL, src.stat().st_size)
    stem = Path(MODEL).stem
Path("/content/stem.txt").write_text(stem, encoding="utf-8")
print("outputs", stem + "_quad.obj", stem + "_color.obj", flush=True)


## Color guide

Stored vertex colors (`COLOR_0`) are used even when every UV coordinate is the same point. A UV channel alone does not hide those colors. If the file has only a base-color texture, the next cell paints that texture onto the vertices and writes `/content/in_guide.glb`. If it has neither, the mesh is used as geometry only. The vertex-color step reads `/content/in.glb`. It is skipped when there is nothing to copy.


In [ ]:
import numpy as np
import trimesh
from pathlib import Path

src = Path("/content/in.glb")
print("color guide: read", src.name, flush=True)
import sys
sys.path.insert(0, "/content/brief153")
from gnn_quad_retopo.glb_vertex_color import load_glb_vertex_colors
loaded = trimesh.load(src, process=False)
_pos, rgb, has_file = load_glb_vertex_colors(src)
geoms = list(loaded.geometry.values()) if isinstance(loaded, trimesh.Scene) else [loaded]
if has_file:
    for geom in geoms:
        if isinstance(geom, trimesh.Trimesh) and len(geom.vertices) == len(rgb):
            rgba = np.full((len(rgb), 4), 255, np.uint8)
            rgba[:, :3] = np.clip(np.round(rgb * 255.0), 0, 255).astype(np.uint8)
            geom.visual = trimesh.visual.ColorVisuals(mesh=geom, vertex_colors=rgba)
            print("reread vertex colors", len(rgb))
parts = [g for g in geoms if isinstance(g, trimesh.Trimesh) and len(g.faces)]
loaded = parts[0] if len(parts) == 1 else trimesh.util.concatenate(parts)

def has_color(mesh):
    vis = getattr(mesh, "visual", None)
    cols = getattr(vis, "vertex_colors", None) if vis is not None else None
    if cols is None or len(cols) != len(mesh.vertices):
        return False
    rgb = np.asarray(cols)[:, :3].astype(np.float32)
    if rgb.size == 0:
        return False
    if rgb.max() > 1.5:
        rgb = rgb / 255.0
    return float(rgb.max() - rgb.min()) >= 1e-3

def sample_texture(img, uv):
    pic = np.asarray(img)
    if pic.ndim == 2:
        pic = pic[..., None]
    if pic.ndim != 3:
        return None
    h, w = pic.shape[:2]
    u = np.clip(uv[:, 0], 0, 0.999)
    v = np.clip(1.0 - uv[:, 1], 0, 0.999)
    x = (u * (w - 1)).astype(np.int32)
    y = (v * (h - 1)).astype(np.int32)
    return pic[y, x, :3]

guide = src
vis = getattr(loaded, "visual", None)
uv = getattr(vis, "uv", None) if vis is not None else None
mat = getattr(vis, "material", None) if vis is not None else None
img = None
if mat is not None:
    img = getattr(mat, "baseColorTexture", None) or getattr(mat, "image", None)
if has_color(loaded):
    print("vertex colors already present")
elif uv is None or img is None:
    print("no vertex colors and no base-color texture; the color term stays off")
else:
    uv = np.asarray(uv, dtype=np.float64)
    samples = sample_texture(img, uv)
    rgb = None
    if samples is not None and len(uv) == len(loaded.vertices):
        rgb = samples
    elif samples is not None and len(uv) == len(loaded.faces) * 3:
        rgb = np.zeros((len(loaded.vertices), samples.shape[1]), dtype=np.float64)
        acc = np.zeros(len(loaded.vertices), dtype=np.float64)
        flat = np.asarray(loaded.faces, dtype=np.int64).reshape(-1)
        np.add.at(rgb, flat, samples)
        np.add.at(acc, flat, 1.0)
        ok = acc > 0
        rgb[ok] /= acc[ok, None]
    if rgb is None:
        print("UV count matches neither vertices nor corners; the color term stays off")
    else:
        if rgb.dtype != np.uint8:
            rgb = np.clip(rgb, 0, 255).astype(np.uint8)
        alpha = np.full((len(rgb), 1), 255, np.uint8)
        loaded.visual = trimesh.visual.ColorVisuals(mesh=loaded, vertex_colors=np.hstack([rgb, alpha]))
        guide = Path("/content/in_guide.glb")
        loaded.export(guide)
        print("wrote", guide.name, guide.stat().st_size)
if has_color(loaded):
    kind = "color"
elif guide != src:
    kind = "texture"
else:
    kind = "mesh"
Path("/content/guide_path.txt").write_text(str(guide), encoding="utf-8")
Path("/content/appearance.txt").write_text(kind, encoding="utf-8")
print("appearance", kind)


## Train and extract

`EPOCHS` is the maximum number of training passes. The run can stop earlier. `TARGET_QUADS = 0` picks a quad count from the mesh Instant Meshes actually receives, after the 790,000-face cap, not from the original face count. `CREASE` is the Instant Meshes crease angle in degrees. `KEEP_DETAIL` does not turn off that cap.

The next cell keeps a **now** line at the top: the current stage, then each epoch. The full log scrolls under it. The log line `fallback` counts only the leftover faces that became three quads each. Those faces do not follow the cross field. If Instant Meshes returns no faces, the run stops instead of splitting the whole mesh. The all-quad mesh is `{name}_quad.obj`, using the input file name.


In [ ]:
import os, shutil, subprocess, sys
from pathlib import Path
import torch
from IPython.display import Markdown, display
EPOCHS = 200  #@param {type:"integer"}
TARGET_QUADS = 0  #@param {type:"integer"}
CREASE = 75  #@param {type:"number"}
KEEP_DETAIL = True  #@param {type:"boolean"}
root = Path("/content/brief153")
work = Path("/content/work")
guide = Path(Path("/content/guide_path.txt").read_text(encoding="utf-8").strip())
if work.exists():
    shutil.rmtree(work)
env = os.environ.copy()
env["PYTHONPATH"] = str(root) + os.pathsep + env.get("PYTHONPATH", "")
env["PYTHONUNBUFFERED"] = "1"
env["PYTHONIOENCODING"] = "utf-8"
dev = "cuda" if torch.cuda.is_available() else "cpu"
cmd = [sys.executable, "-u", "-m", "gnn_quad_retopo.cli", "run", str(guide),
       "--out", str(work), "--extractor", "im", "--vertex-color-guide",
       "--epochs", str(int(EPOCHS)), "--target-quads", str(int(TARGET_QUADS)),
       "--im-crease-angle", str(float(CREASE)), "--auto", "--device", dev]
if KEEP_DETAIL:
    cmd.append("--no-reg-decimate")
print("run: start", " ".join(cmd), flush=True)
status = display(Markdown("**now:** starting"), display_id=True)
proc = subprocess.Popen(
    cmd, cwd=str(root), env=env,
    stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
    text=True, encoding="utf-8", errors="replace", bufsize=1,
)
assert proc.stdout is not None
for line in proc.stdout:
    print(line, end="", flush=True)
    text = line.strip()
    if "[PIPELINE] Stage" in text or "[PIPELINE] ALL DONE" in text or "[PIPELINE] FAILED" in text:
        status.update(Markdown("**now:** " + text))
    elif " epoch " in text and "total=" in text:
        status.update(Markdown("**now:** " + text))
    elif text.startswith("[im] ") or text.startswith("[run] "):
        status.update(Markdown("**now:** " + text))
code = proc.wait()
if code != 0:
    status.update(Markdown("**now:** failed (exit %d)" % code))
    raise SystemExit(code)
quads = list(work.rglob("retopologized_quad_mesh.obj"))
assert quads, "No quad OBJ was written."
stem = Path("/content/stem.txt").read_text(encoding="utf-8").strip()
quad_path = Path("/content") / (stem + "_quad.obj")
shutil.copyfile(quads[-1], quad_path)
nbytes = quad_path.stat().st_size
print("run: quad", quad_path.name, nbytes, flush=True)
status.update(Markdown("**now:** quad OBJ written (%d bytes)" % nbytes))


## Vertex color

Copies color from the original GLB onto the quad vertices. Writes `{name}_color.obj` as quads with `v x y z r g b`. No materials and no textures. For PBR, bake in Blender yourself from this OBJ and the original GLB. A mesh with neither is left as `{name}_quad.obj` only.


In [ ]:
import sys
from pathlib import Path
print("vertex color: start", flush=True)
sys.path.insert(0, "/content/brief153")
kind = Path("/content/appearance.txt").read_text(encoding="utf-8").strip()
if kind == "mesh":
    print("vertex color: skipped (no color on the source)", flush=True)
else:
    from gnn_quad_retopo.vertex_color import write_quad_vertex_colors
    stem = Path("/content/stem.txt").read_text(encoding="utf-8").strip()
    quad_path = Path("/content") / (stem + "_quad.obj")
    color_path = Path("/content") / (stem + "_color.obj")
    write_quad_vertex_colors(quad_path, "/content/in.glb", color_path)
    print("vertex color: wrote", color_path.name, color_path.stat().st_size, flush=True)


## Viewer

Drag to orbit. **Wireframe** shows the quad edges. **Vertex color** shows the colors on the mesh. There is no texture. A large mesh is slow in the browser.


In [ ]:
import shutil, socket, subprocess, sys
from pathlib import Path
from google.colab import output
view = Path("/content/view")
if view.exists():
    shutil.rmtree(view)
shutil.copytree("/content/brief153/viewer", view)
stem = Path("/content/stem.txt").read_text(encoding="utf-8").strip()
shutil.copyfile(Path("/content") / (stem + "_quad.obj"), view / "quad.obj")
color_path = Path("/content") / (stem + "_color.obj")
if color_path.is_file():
    shutil.copyfile(color_path, view / "color.obj")
port = 8765
s = socket.socket()
s.settimeout(0.3)
try:
    s.connect(("127.0.0.1", port))
    running = True
except OSError:
    running = False
finally:
    s.close()
if not running:
    subprocess.Popen([sys.executable, "-m", "http.server", str(port), "--bind", "127.0.0.1"], cwd=str(view))
output.serve_kernel_port_as_iframe(port, height="640")


## Download

The zip is named after the input file. It holds `{name}_quad.obj` and `{name}_color.obj`. Keep the quad file when the next tool should see only quads. The color file is the same quads with vertex colors. Bake PBR yourself in Blender from that OBJ and the original GLB.


In [ ]:
import zipfile
from pathlib import Path
from google.colab import files
stem = Path("/content/stem.txt").read_text(encoding="utf-8").strip()
out = Path("/content") / (stem + ".zip")
names = [stem + "_quad.obj", stem + "_color.obj"]
with zipfile.ZipFile(out, "w", zipfile.ZIP_DEFLATED) as z:
    for name in names:
        p = Path("/content") / name
        if p.is_file():
            z.write(p, name)
print(out, out.stat().st_size)
files.download(str(out))
